# SpatialDataset Presentation in Notebooks

`SpatialDataset` is the bridge between geospatial data and Lets-Plot map layers. In Kotlin notebooks it is also convenient to inspect it directly: when a `SpatialDataset` is the last expression in a cell, it is displayed as a compact table.

In [1]:
%useLatestDescriptors
%use lets-plot

In [2]:
LetsPlot.getInfo()

Lets-Plot Kotlin API v.0.0.0-SNAPSHOT. Frontend: Notebook with dynamically loaded JS. Lets-Plot JS v.4.11.0rc1.
Outputs: Web (HTML+JS), Static SVG (hidden)

## Small Spatial Dataset

The dataset below contains a few landmarks represented by GeoJSON point geometries. The attribute columns (`name`, `kind`, and `visitors`) can be used both for inspection and for plot mappings.

In [3]:
val places = SpatialDataset.withGEOJSON(
    data = mapOf(
        "name" to listOf("Museum", "Garden", "Harbor", "Library", "Market"),
        "kind" to listOf("culture", "park", "waterfront", "culture", "food"),
        "visitors" to listOf(120, 95, 80, 65, 140)
    ),
    geometry = listOf(
        "{\"type\":\"Point\",\"coordinates\":[-73.9857,40.7484]}",
        "{\"type\":\"Point\",\"coordinates\":[-73.9654,40.7829]}",
        "{\"type\":\"Point\",\"coordinates\":[-74.0122,40.7061]}",
        "{\"type\":\"Point\",\"coordinates\":[-73.9822,40.7532]}",
        "{\"type\":\"Point\",\"coordinates\":[-73.9969,40.7223]}"
    )
)

places

name,kind,visitors,geometry
Museum,culture,120,POINT (-73.9857 40.7484)
Garden,park,95,POINT (-73.9654 40.7829)
Harbor,waterfront,80,POINT (-74.0122 40.7061)
Library,culture,65,POINT (-73.9822 40.7532)
Market,food,140,POINT (-73.9969 40.7223)


The inspected object can be passed directly to map-aware geometry layers via the `map` parameter.

In [4]:
letsPlot() +
    geomPoint(map = places) {
        color = "kind"
        size = "visitors"
    } +
    geomText(map = places, nudgeX = 0.01, hjust = "left") {
        label = "name"
    } +
    scaleSize(range = 5 to 12) +
    labs(color = "place type", size = "visitors") +
    ggtitle("Landmarks from a SpatialDataset") +
    ggsize(650, 420) +
    themeVoid()

Museum 
 
 
 
 
 
 
 Garden 
 
 
 
 
 
 
 Harbor 
 
 
 
 
 
 
 Library 
 
 
 
 
 
 
 Market 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 Landmarks from a SpatialDataset 
 
 
 
 
 
 
 
 
 visitors 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 80 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 100 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 120 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 140 
 
 
 
 
 
 
 
 
 
 
 
 
 place type 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 culture 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 park 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 waterfront 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 food

## Large Datasets

Notebook output is capped to the first 5 rows, so a large `SpatialDataset` remains easy to inspect without flooding the notebook.

In [5]:
val n = 25
val manyPlaces = SpatialDataset.withGEOJSON(
    data = mapOf(
        "id" to (1..n).toList(),
        "group" to (1..n).map { if (it % 2 == 0) "even" else "odd" }
    ),
    geometry = (1..n).map {
        "{\"type\":\"Point\",\"coordinates\":[${-74.05 + it * 0.005},${40.70 + it * 0.004}]}"
    }
)

manyPlaces

id,group,geometry
1,odd,POINT (-74.045 40.704)
2,even,POINT (-74.04 40.708)
3,odd,POINT (-74.035 40.712)
4,even,POINT (-74.03 40.716)
5,odd,POINT (-74.025 40.72)
6,even,POINT (-74.02 40.724)
7,odd,POINT (-74.015 40.728)
8,even,POINT (-74.01 40.732)
9,odd,POINT (-74.005 40.736)
10,even,POINT (-74 40.74)


## Inspect a Real-World Dataset

The examples above used a few inline GeoJSON points. A `SpatialDataset` can also wrap real geospatial data loaded through [GeoTools](https://geotools.org/). Below we load the *Natural Earth* countries dataset — 177 countries with polygon borders — and inspect it as a table before drawing it on a map.

In [6]:
@file:Repository("https://repo.osgeo.org/repository/release/")
@file:DependsOn("org.geotools:gt-shapefile:33.2")
@file:DependsOn("org.geotools:gt-epsg-hsql:33.2")

In [7]:
%use lets-plot-gt

In [8]:
import org.geotools.data.shapefile.ShapefileDataStoreFactory
import java.net.URL

// Load the Natural Earth "low resolution" countries shapefile and convert it to a SpatialDataset.
val shpUrl = "https://raw.githubusercontent.com/JetBrains/lets-plot-kotlin/master/docs/examples/shp/naturalearth_lowres/naturalearth_lowres.shp"
val countries = ShapefileDataStoreFactory()
    .createDataStore(URL(shpUrl))
    .featureSource.features
    .toSpatialDataset()

In [9]:
countries

pop_est,continent,name,iso_a3,gdp_md_est,geometry
920938,Oceania,Fiji,FJI,8374,"MULTIPOLYGON (((180 -16.0671326636, 180 -16.5552165666, 179.364142662 -16.8013540769, ...)), ((178.12557 -17.50481, 178.3736 -17.33992, 178.71806 -17.62846, ...)), ((-179.793320109 -16.0208822567, -179.9173693848 -16.5017831356, -180 -16.5552165666, ...)))"
53950935,Africa,Tanzania,TZA,150600,"MULTIPOLYGON (((33.9037111971 -0.95, 34.07262 -1.05982, 37.69869 -3.09699, ...)))"
603253,Africa,W. Sahara,ESH,906.5,"MULTIPOLYGON (((-8.6655895655 27.6564258896, -8.6651244776 27.5894790716, -8.6843997868 27.3957441269, ...)))"
35623680,North America,Canada,CAN,1674000,"MULTIPOLYGON (((-122.84 49, -122.97421 49.0025377778, -124.91024 49.98456, ...)), ((-83.99367 62.4528, -83.25048 62.91409, -81.87699 62.90458, ...)), ((-79.7758331299 72.8029022217, -80.8760986328 73.3331832886, -80.8338851929 73.6931838989, ...)), ...)"
326625791,North America,United States of America,USA,18560000,"MULTIPOLYGON (((-122.84 49, -120 49, -117.03121 49, ...)), ((-155.40214 20.07975, -155.22452 19.99302, -155.06226 19.8591, ...)), ((-155.99566 20.76404, -156.07926 20.64397, -156.41445 20.57241, ...)), ...)"
18556698,Asia,Kazakhstan,KAZ,460700,"MULTIPOLYGON (((87.3599703308 49.2149807806, 86.5987764831 48.549181627, 85.7682328633 48.4557506374, ...)))"
29748859,Asia,Uzbekistan,UZB,202300,"MULTIPOLYGON (((55.9681913593 41.3086416693, 55.9289172707 44.9958584662, 58.5031270689 45.5868043076, ...)))"
6909701,Oceania,Papua New Guinea,PNG,28020,"MULTIPOLYGON (((141.0002104026 -2.6001510555, 142.7352466168 -3.2891529273, 144.583970982 -3.8614177385, ...)), ((152.6400167177 -3.6599830054, 153.0199935244 -3.9800151506, 153.1400378766 -4.4999834123, ...)), ((151.3013904157 -5.8407284481, 150.7544470563 -6.0837627092, 150.2411967308 -6.3177535946, ...)), ...)"
260580739,Asia,Indonesia,IDN,3028000,"MULTIPOLYGON (((141.0002104026 -2.6001510555, 141.0170569195 -5.8590219051, 141.03385176 -9.1178927548, ...)), ((124.9686824891 -8.8927902157, 125.0700199728 -9.0899874813, 125.0885201356 -9.3931731096, ...)), ((134.2101339052 -6.8952377255, 134.1127755067 -6.1424671363, 134.2903357281 -5.7830575497, ...)), ...)"
44293293,South America,Argentina,ARG,879400,"MULTIPOLYGON (((-68.6340102276 -52.6363704589, -68.25 -53.1, -67.75 -53.85, ...)), ((-57.6251334296 -30.2162948545, -57.8749373033 -31.0165560849, -58.142440355 -32.0445036761, ...)))"


The table shows the attribute columns (`pop_est`, `continent`, `name`, `iso_a3`, `gdp_md_est`) alongside the `geometry` column, where each country's `Polygon` / `MultiPolygon` border is pretty-printed as WKT. Long borders are abbreviated — only the first few vertices of each ring are shown, followed by `...` — and only the first 5 of 177 rows are displayed.

The number of previewed rows is configurable; set it to 10 to inspect a larger preview:

In [10]:
letsPlotNotebookConfig.spatialDatasetRowLimit = 10
countries

pop_est,continent,name,iso_a3,gdp_md_est,geometry
920938,Oceania,Fiji,FJI,8374,"MULTIPOLYGON (((180 -16.0671326636, 180 -16.5552165666, 179.364142662 -16.8013540769, ...)), ((178.12557 -17.50481, 178.3736 -17.33992, 178.71806 -17.62846, ...)), ((-179.793320109 -16.0208822567, -179.9173693848 -16.5017831356, -180 -16.5552165666, ...)))"
53950935,Africa,Tanzania,TZA,150600,"MULTIPOLYGON (((33.9037111971 -0.95, 34.07262 -1.05982, 37.69869 -3.09699, ...)))"
603253,Africa,W. Sahara,ESH,906.5,"MULTIPOLYGON (((-8.6655895655 27.6564258896, -8.6651244776 27.5894790716, -8.6843997868 27.3957441269, ...)))"
35623680,North America,Canada,CAN,1674000,"MULTIPOLYGON (((-122.84 49, -122.97421 49.0025377778, -124.91024 49.98456, ...)), ((-83.99367 62.4528, -83.25048 62.91409, -81.87699 62.90458, ...)), ((-79.7758331299 72.8029022217, -80.8760986328 73.3331832886, -80.8338851929 73.6931838989, ...)), ...)"
326625791,North America,United States of America,USA,18560000,"MULTIPOLYGON (((-122.84 49, -120 49, -117.03121 49, ...)), ((-155.40214 20.07975, -155.22452 19.99302, -155.06226 19.8591, ...)), ((-155.99566 20.76404, -156.07926 20.64397, -156.41445 20.57241, ...)), ...)"


The inspected `SpatialDataset` can be drawn directly with `geomMap`, colored by an attribute column.

In [11]:
letsPlot() +
    geomMap(map = countries) { fill = "continent" } +
    labs(fill = "continent") +
    ggtitle("Natural Earth countries") +
    ggsize(800, 450) +
    themeVoid()